In [0]:
# Databricks notebook source
from pyspark.sql.functions import col, trim, to_date

df_meds_bronze = spark.read.table("workspace.bronze.medications")

df_meds_silver = (
    df_meds_bronze
    .filter(col("patient").isNotNull() & col("code").isNotNull())
    .select(
        to_date(col("start"), "yyyy-MM-dd").alias("start_date"),
        to_date(col("stop"), "yyyy-MM-dd").alias("stop_date"),
        col("patient").alias("patient_id"),
        col("encounter").alias("encounter_id"),
        col("code").alias("medication_code"),
        trim(col("description")).alias("medication_description"),
        col("reasoncode").alias("reason_code"),
        trim(col("reasondescription")).alias("reason_description"),
        col("ingested_at")
    )
)

(
    df_meds_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.medications")
)

print(f"✅ Created workspace.silver.medications with {df_meds_silver.count()} clean rows!")